# 1.RunnableWithMessageHistory使用示例

知识点讲解：RunnableWithMessageHistory 简化多轮对话代码

1. RunnableWithMessageHistory 的设计定位
   - 是 LangChain 1.x 推荐的记忆（Memory）管理方案，替代 0.x 的 Memory 组件
   - 本质是一个装饰器（Wrapper）：包装任意 Runnable，为其注入历史读写逻辑
   - 遵循"显式优于隐式"原则，历史存储由开发者通过回调函数掌控

2. 它自动完成的三件事
   - 读取历史：调用 get_session_history(session_id) 拿到历史消息列表
   - 注入历史：把消息列表填入 history_messages_key 指定的变量
   - 写回历史：执行完成后自动追加本轮的 HumanMessage 与 AIMessage

3. 与直接手写历史管理的对比
   - 手写方式需要在每次调用前后手动 append 消息，容易遗漏且代码重复
   - RunnableWithMessageHistory 将这部分模板代码收敛到框架内部
   - 同时保留了存储实现的自由度（内存/文件/Redis/数据库）

4. FileChatMessageHistory 的特点
   - 将消息以 JSON 格式序列化写入本地文本文件
   - 优点：跨进程持久化，程序重启后对话历史依然可用
   - 缺点：无并发控制，不适合高并发生产环境（生产推荐 Redis/SQL）

5. 流式输出与历史写回的配合
   - 使用 stream() 时，框架会先逐块 yield 内容给调用方
   - 流结束后（生成器耗尽时）才把完整的 AI 回复写回历史
   - 因此必须完整消费生成器，否则历史可能不完整

6. 关键参数的对应关系（三处必须一致）
   - prompt 中的 MessagesPlaceholder("history")
   - RunnableWithMessageHistory 的 history_messages_key="history"
   - prompt 中的 {query} 与 input_messages_key="query"

7. 数据流转过程（单轮）
   - {"query": 用户输入} + config{session_id}
   - → 读取 FileChatMessageHistory.messages → list[BaseMessage]
   - → {"query": ..., "history": [...]} → prompt → llm → StrOutputParser → str
   - → 追加写回：HumanMessage(query) + AIMessage(完整回复)


In [ ]:
import dotenv
from langchain_community.chat_message_histories import FileChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
store = {}

In [ ]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """根据 session_id 获取（或创建）对应的文件型消息历史对象
    参数：
        session_id: str - 会话唯一标识，由 config.configurable.session_id 注入
    返回：
        BaseChatMessageHistory - 具备 messages 读属性与 add_message 写方法的历史对象
    调用时机：
        RunnableWithMessageHistory 在每次 invoke/stream 时自动调用
    FileChatMessageHistory(file_path: str) -> FileChatMessageHistory
      作用：创建基于本地文件的消息历史存储
      参数：file_path - 历史文件路径，不存在时自动创建
      返回：FileChatMessageHistory 实例
      存储格式：JSON 数组，每个元素为序列化后的 BaseMessage
      特点：进程重启后历史依然保留，适合单机演示与轻量场景
    """
    if session_id not in store:
        store[session_id] = FileChatMessageHistory(f"./chat_history_{session_id}.txt")
    return store[session_id]

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个强大的聊天机器人，请根据用户的需求回复问题。"),
    MessagesPlaceholder("history"),
    ("human", "{query}"),
])
llm = ChatOpenAI(model="deepseek-v4-pro")
chain = prompt | llm | StrOutputParser()
with_message_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="query",
    history_messages_key="history",
)
while True:
    query = input("Human: ")
    if query == "q":
        exit(0)
    response = with_message_chain.stream(
        {"query": query},
        config={"configurable": {"session_id": "muxiaoke"}}
    )
    print("AI: ", flush=True, end="")
    for chunk in response:
        print(chunk, flush=True, end="")
    print("")